# 03 · Offline (open-loop) evaluation on recorded demos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/vla_ros2/blob/main/notebooks/03_offline_eval.ipynb)

At every N-th frame of a recorded `lerobot/svla_so100_stacking` episode the policy
predicts a 50-step action chunk from the recorded cameras and joints; the chunk is
compared with what the demonstrator actually did next. Reported beside it is a
**hold-state baseline** (keep the current joints): a policy that does not beat it
has not learned the motion.

This is imitation error, not task success, and it is only a held-out number on
episodes the checkpoint did not train on.

Measured on CPU (2026-10-11), episode 0, every 150th frame: `smolvla_base`
MAE **39.8°** vs hold-state **4.6°**. `smolvla_base` outputs ±1-scale normalized
actions and expects `camera1..3`, so it needs fine-tuning before it is meaningful here.

In [ ]:
# Colab: clone the repo. Locally: run from the repo (or its notebooks/ folder).
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("vla_ros2").exists():
    !git clone --depth 1 https://github.com/rsasaki0109/vla_ros2.git
REPO = Path("vla_ros2").resolve() if IN_COLAB else Path.cwd()
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
%pip install -q -e ".[smolvla]"

In [ ]:
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Choose checkpoints

Add your fine-tuned checkpoint (a Hugging Face repo id or a local
`.../pretrained_model` path). Hold out episodes it was not trained on.

In [ ]:
CHECKPOINTS = {
    "smolvla_base": "lerobot/smolvla_base",
    # "finetuned": "your-hf-user/smolvla_so100_stacking",
}
EPISODES = [50, 51, 52, 53, 54, 55]  # keep these out of fine-tuning to get a held-out number
STRIDE = 60

In [ ]:
import json
import subprocess


def run(cmd):
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-3000:])
    return proc.stdout


results = {}
for label, pretrained in CHECKPOINTS.items():
    out = f"/tmp/offline_eval_{label}.json"
    cmd = [
        sys.executable, "scripts/eval_offline_lerobot.py",
        "--pretrained", pretrained, "--device", DEVICE,
        "--episodes", *map(str, EPISODES), "--stride", str(STRIDE), "--out", out,
    ]
    print(run(cmd))
    results[label] = json.load(open(out))

In [ ]:
print(f"{'checkpoint':<16}{'frames':>7}{'MAE':>9}{'1st-step':>10}{'hold-state':>12}{'ms/chunk':>10}")
for label, r in results.items():
    base = r["hold_state_baseline_errors"]
    print(
        f"{label:<16}{r['num_frames']:>7}{r['model_errors']['mae']:>9.2f}"
        f"{r['model_errors']['first_step_mae']:>10.2f}{base['mae']:>12.2f}{r['infer_ms_mean']:>10.0f}"
    )